In [1]:
import os
import json
import numpy as np

In [2]:
def extract_comprehensive_features(trace_events):
    features = {}
    
    # Filter by type
    keystrokes = [e for e in trace_events if e['type'] in ['keydown', 'keypress']]
    clicks = [e for e in trace_events if e['type'] == 'click']
    scrolls = [e for e in trace_events if e['type'] == 'scroll']
    navigates = [e for e in trace_events if e['type'] == 'navigate']
    
    # KEYSTROKE FEATURES
    latencies = [e.get('latency', 0) for e in keystrokes if e.get('latency')]
    if latencies:
        features['mean_keystroke_latency'] = np.mean(latencies)
        features['std_keystroke_latency'] = np.std(latencies)
        features['max_keystroke_latency'] = np.max(latencies)
        features['min_keystroke_latency'] = np.min(latencies)
    
    # CLICK FEATURES
    click_latencies = [e.get('latency', 0) for e in clicks if e.get('latency')]
    if click_latencies:
        features['mean_click_interval'] = np.mean(click_latencies)
        features['std_click_interval'] = np.std(click_latencies)
    
    click_positions_x = [e.get('x', 0) for e in clicks]
    if click_positions_x:
        features['click_std_x'] = np.std(click_positions_x)
    
    # SCROLL FEATURES
    scroll_distances = [abs(e.get('scrollX', 0)) + abs(e.get('scrollY', 0)) for e in scrolls]
    if scroll_distances:
        features['mean_scroll_distance'] = np.mean(scroll_distances)
        features['scroll_frequency'] = len(scrolls) / (trace_events[-1]['relativeTime'] / 1000 / 60)  # per minute
    
    # TIMING FEATURES
    features['total_duration'] = trace_events[-1]['relativeTime']
    features['num_actions'] = len(keystrokes) + len(clicks) + len(scrolls)
    
    # ACTION RATIOS
    total_actions = features['num_actions']
    if total_actions > 0:
        features['keystroke_ratio'] = len(keystrokes) / total_actions
        features['click_ratio'] = len(clicks) / total_actions
        features['scroll_ratio'] = len(scrolls) / total_actions
    
    return features

In [4]:
DATA_PATH = "../artifacts/20260328_005930_992/trace.jsonl"
with open(DATA_PATH) as f:
    trace = [json.loads(line) for line in f]

print(f"Loaded {len(trace)} events")
trace[0]


Loaded 234 events


{'timestamp': 1774659604171.9092,
 'type': 'navigate',
 'relativeTime': 18.18701171875,
 'url': 'https://www.wikipedia.org/',
 'origin': 'www.wikipedia.org',
 'reason': 'initial',
 'pageLoadTime': 355.0}

In [5]:
feats = extract_comprehensive_features(trace)

In [6]:
feats

{'mean_keystroke_latency': np.float64(324.4791666666667),
 'std_keystroke_latency': np.float64(1263.041269937753),
 'max_keystroke_latency': np.int64(6088),
 'min_keystroke_latency': np.int64(1),
 'click_std_x': np.float64(0.0),
 'mean_scroll_distance': np.float64(428.0),
 'scroll_frequency': 113.9000210925965,
 'total_duration': 4741,
 'num_actions': 231,
 'keystroke_ratio': 0.9567099567099567,
 'click_ratio': 0.004329004329004329,
 'scroll_ratio': 0.03896103896103896}